# Phase 5 - Layer E: Candidate-Slot Generation and Weights

This notebook is the bridge from the realistic data pipeline back into the original optimization architecture.

Layer C produces raw physical visibility windows. Layer D enriches those windows with operational assumptions. Layer E explicitly performs the original discretization idea: each enriched visibility window can produce multiple fixed-length candidate slots.

Priority, urgency, normalized metrics, and final weights are assigned at the parent visibility-window level and then inherited by all child candidate slots. This keeps the operational value attached to the communication opportunity, while timing alternatives matter through feasibility and conflicts downstream.

The final optimization units are still candidate slots, preserving compatibility with the reduction, conflict-graph, MWIS, and QUBO workflow.

## Configuration

All Layer E files are kept in this `add-weight-data` folder. `MIN_CONTACT_DURATION_MINUTES` is the minimum useful scheduled contact duration, not the physical visibility duration. `STEP_SIZE_MINUTES` controls how densely candidate alternatives are generated inside each enriched visibility window.

For this first implementation, every generated candidate slot has the same duration within the scenario. Longer visibility windows therefore create more alternatives, not longer contacts. The duration term in the weight formula is kept for compatibility and future extension, but it is less informative while contact duration is fixed.

In [1]:
from pathlib import Path
import hashlib
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = ["pandas"]
missing_packages = [
    package_name
    for package_name in REQUIRED_PACKAGES
    if importlib.util.find_spec(package_name) is None
]

if missing_packages:
    print(f"Installing missing packages into this notebook kernel: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Dependency installation finished. If imports still fail, restart the kernel and run the notebook again.")

import pandas as pd

MIN_CONTACT_DURATION_MINUTES = 5
STEP_SIZE_MINUTES = 2
INCLUDE_UNAVAILABLE_WINDOWS = False
PRIORITY_MODE = "satellite_stable_with_pass_quality"
URGENCY_MODE = "earlier_slots_more_urgent"
PRIORITY_RANDOM_SEED = 20260531

SMALL_INSTANCE_SIZE = 100
MEDIUM_INSTANCE_SIZE = 500
DENSE_INSTANCE_SIZE = 700
DENSE_STATION_COUNT = 6

PRIORITY_WEIGHT = 0.45
DURATION_WEIGHT = 0.25
URGENCY_WEIGHT = 0.20
ELEVATION_WEIGHT = 0.10

DATA_DIR = Path("../../data/dataset/starlink")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OPERATIONAL_INPUT_CSV = DATA_DIR / "starlink_visibility_windows_operational.csv"
FULL_OUTPUT_CSV = DATA_DIR / "starlink_scheduling_slots_weighted.csv"
SMALL_OUTPUT_CSV = DATA_DIR / "starlink_scheduling_instance_small.csv"
MEDIUM_OUTPUT_CSV = DATA_DIR / "starlink_scheduling_instance_medium.csv"
DENSE_OUTPUT_CSV = DATA_DIR / "starlink_scheduling_instance_dense.csv"

MIN_CONTACT_DURATION_SECONDS = MIN_CONTACT_DURATION_MINUTES * 60
STEP_SIZE_SECONDS = STEP_SIZE_MINUTES * 60
if STEP_SIZE_SECONDS <= 0 or MIN_CONTACT_DURATION_SECONDS <= 0:
    raise ValueError("Contact duration and step size must be positive.")
if STEP_SIZE_SECONDS > MIN_CONTACT_DURATION_SECONDS:
    raise ValueError("STEP_SIZE_MINUTES should be less than or equal to MIN_CONTACT_DURATION_MINUTES.")

print(f"Notebook Python executable: {sys.executable}")
print(f"Layer E data directory: {DATA_DIR.resolve()}")
print(f"Operational input: {OPERATIONAL_INPUT_CSV.resolve()}")
print(f"Minimum useful contact duration: {MIN_CONTACT_DURATION_MINUTES} minutes")
print(f"Discretization step size: {STEP_SIZE_MINUTES} minutes")
print(f"Full candidate-slot output: {FULL_OUTPUT_CSV.resolve()}")

Notebook Python executable: d:\University\PoliMi\courses\SWE2\SWE2-research-project\.venv\Scripts\python.exe
Layer E data directory: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink
Operational input: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_visibility_windows_operational.csv
Minimum useful contact duration: 5 minutes
Discretization step size: 2 minutes
Full candidate-slot output: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_slots_weighted.csv


## Load Enriched Visibility Windows

Layer E starts from the Layer D enriched visibility-window table. By default, windows that overlap synthetic blackouts are excluded before candidate-slot generation.

In [2]:
def parse_boolean_series(values: pd.Series, column_name: str) -> pd.Series:
    """Parse a true/false CSV column without relying on pandas dtype inference.

    Parameters
    ----------
    values:
        Raw column values read from CSV.
    column_name:
        Name used in validation errors.

    Returns
    -------
    pandas.Series
        Boolean series with True and False values.
    """
    parsed = values.astype(str).str.strip().str.lower().map({"true": True, "false": False})
    if parsed.isna().any():
        raise ValueError(f"Could not parse all {column_name} values as true/false.")
    return parsed


def load_enriched_visibility_windows(input_csv_path: Path, include_unavailable: bool = False) -> pd.DataFrame:
    """Load Layer D enriched visibility windows for candidate-slot generation.

    Parameters
    ----------
    input_csv_path:
        Path to the Layer D enriched visibility-window CSV.
    include_unavailable:
        When True, keep windows flagged as operationally unavailable.

    Returns
    -------
    pandas.DataFrame
        Enriched visibility windows with parsed UTC timestamps.
    """
    if not input_csv_path.exists():
        raise FileNotFoundError(f"Missing Layer D operational CSV: {input_csv_path}")

    windows_df = pd.read_csv(input_csv_path)
    required_columns = [
        "window_id",
        "satellite_name",
        "norad_id",
        "station_id",
        "station_name",
        "start_time",
        "end_time",
        "duration_seconds",
        "max_elevation_deg",
        "source_epoch",
        "antenna_type",
        "station_capacity",
        "reconfiguration_time_seconds",
        "is_operationally_available",
    ]
    missing_columns = [column for column in required_columns if column not in windows_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required Layer D columns: {missing_columns}")

    windows_df = windows_df.copy()
    windows_df["start_time"] = pd.to_datetime(windows_df["start_time"], utc=True)
    windows_df["end_time"] = pd.to_datetime(windows_df["end_time"], utc=True)
    windows_df["is_operationally_available"] = parse_boolean_series(
        windows_df["is_operationally_available"],
        "is_operationally_available",
    )

    if not include_unavailable:
        windows_df = windows_df.loc[windows_df["is_operationally_available"]].copy()

    windows_df = windows_df.sort_values(["start_time", "station_id", "norad_id"]).reset_index(drop=True)
    return windows_df


enriched_windows = load_enriched_visibility_windows(OPERATIONAL_INPUT_CSV, INCLUDE_UNAVAILABLE_WINDOWS)

print(f"Enriched visibility windows loaded for Layer E: {len(enriched_windows)}")
print(f"Unique satellites: {enriched_windows['norad_id'].nunique()}")
print(f"Unique stations: {enriched_windows['station_id'].nunique()}")
print("\nDuration summary for enriched windows, seconds:")
print(enriched_windows["duration_seconds"].describe().to_string())
enriched_windows.head(10)

Enriched visibility windows loaded for Layer E: 6062
Unique satellites: 96
Unique stations: 100

Duration summary for enriched windows, seconds:
count    6062.000000
mean      301.930056
std        98.836841
min        60.000000
25%       240.000000
50%       300.000000
75%       360.000000
max       480.000000


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,availability_start,availability_end,blackout_window_ids,overlaps_blackout,is_operationally_available,operational_constraint_source
0,VIS_000001,STARLINK-1296,45412,GATEWAY_0001,"Adelanto, CA",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,42.800,2026-05-31T04:29:06Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
1,VIS_000002,STARLINK-1300,45374,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,10.784,2026-05-30T09:25:07Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
2,VIS_000003,STARLINK-1154,45198,GATEWAY_0003,"Akita, Japan",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,15.656,2026-05-30T22:10:10Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
3,VIS_000004,STARLINK-1269,45197,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:02:00+00:00,120,23.820,2026-05-31T00:58:13Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
4,VIS_000005,STARLINK-1350,45535,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,13.228,2026-05-30T21:51:18Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
5,VIS_000006,STARLINK-1039,44744,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31 00:00:00+00:00,2026-05-31 00:07:00+00:00,420,53.684,2026-05-30T09:40:14Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
6,VIS_000007,STARLINK-1284,45390,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31 00:00:00+00:00,2026-05-31 00:05:00+00:00,300,26.758,2026-05-30T03:35:30Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
7,VIS_000008,STARLINK-1263,45405,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31 00:00:00+00:00,2026-05-31 00:04:00+00:00,240,53.290,2026-05-31T02:35:51Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
8,VIS_000009,STARLINK-1269,45197,GATEWAY_0007,"Anderson, SC",2026-05-31 00:00:00+00:00,2026-05-31 00:04:00+00:00,240,68.344,2026-05-31T00:58:13Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions
9,VIS_000010,STARLINK-1350,45535,GATEWAY_0007,"Anderson, SC",2026-05-31 00:00:00+00:00,2026-05-31 00:05:00+00:00,300,26.181,2026-05-30T21:51:18Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,NaN,False,True,deterministic synthetic Layer D assumptions


## E1 - Generate Candidate Slots

Each enriched visibility window is discretized into all fixed-length candidate slots that fit completely inside the parent window. If a parent window is shorter than the minimum useful contact duration, it generates no candidate slots.

In [3]:
def generate_candidate_slots_for_window(
    window: pd.Series,
    min_contact_seconds: int,
    step_seconds: int,
) -> list[dict]:
    """Generate fixed-length candidate slots inside one enriched visibility window.

    Parameters
    ----------
    window:
        One enriched visibility-window record from Layer D.
    min_contact_seconds:
        Fixed minimum useful scheduled contact duration.
    step_seconds:
        Discretization step between candidate start times.

    Returns
    -------
    list[dict]
        Candidate slot records that preserve traceability to the parent window.
    """
    window_start = window["start_time"]
    window_end = window["end_time"]
    window_duration_seconds = int((window_end - window_start).total_seconds())
    if window_duration_seconds < min_contact_seconds:
        return []

    candidate_records = []
    latest_start = window_end - pd.Timedelta(seconds=min_contact_seconds)
    current_start = window_start
    local_index = 1

    while current_start <= latest_start:
        current_end = current_start + pd.Timedelta(seconds=min_contact_seconds)
        antenna_id = f"ANT_{window['station_id']}"
        candidate_records.append(
            {
                "parent_window_id": window["window_id"],
                "source_interval_id": window["window_id"],
                "satellite_name": window["satellite_name"],
                "satellite_id": str(window["norad_id"]),
                "norad_id": int(window["norad_id"]),
                "antenna_id": antenna_id,
                "station_id": window["station_id"],
                "station_name": window["station_name"],
                "start_time": current_start.isoformat(),
                "end_time": current_end.isoformat(),
                "start_time_hours": (current_start - SCENARIO_START).total_seconds() / 3600,
                "end_time_hours": (current_end - SCENARIO_START).total_seconds() / 3600,
                "duration_seconds": min_contact_seconds,
                "duration_hours": min_contact_seconds / 3600,
                "parent_visibility_start": window_start.isoformat(),
                "parent_visibility_end": window_end.isoformat(),
                "parent_visibility_duration_seconds": window_duration_seconds,
                "candidate_index_in_parent": local_index,
                "max_elevation_deg": window["max_elevation_deg"],
                "source_epoch": window["source_epoch"],
                "antenna_type": window["antenna_type"],
                "station_capacity": window["station_capacity"],
                "reconfiguration_time_seconds": window["reconfiguration_time_seconds"],
                "reconfiguration_time_hours": window["reconfiguration_time_seconds"] / 3600,
                "is_operationally_available": window["is_operationally_available"],
            }
        )
        local_index += 1
        current_start += pd.Timedelta(seconds=step_seconds)

    return candidate_records


def generate_candidate_slots(
    windows_df: pd.DataFrame,
    min_contact_seconds: int,
    step_seconds: int,
) -> pd.DataFrame:
    """Generate candidate slots from all enriched visibility windows.

    Parameters
    ----------
    windows_df:
        Layer D enriched visibility-window table.
    min_contact_seconds:
        Fixed scheduled contact duration for each candidate slot.
    step_seconds:
        Discretization step between candidate start times.

    Returns
    -------
    pandas.DataFrame
        Candidate-slot table before priority, urgency, and weight assignment.
    """
    records = []
    for _, window in windows_df.iterrows():
        records.extend(generate_candidate_slots_for_window(window, min_contact_seconds, step_seconds))

    if not records:
        raise ValueError("No candidate slots were generated. Try lowering MIN_CONTACT_DURATION_MINUTES.")

    slots_df = pd.DataFrame.from_records(records)
    slots_df = slots_df.sort_values(["start_time", "station_id", "norad_id", "candidate_index_in_parent"]).reset_index(drop=True)
    slots_df.insert(0, "slot_id", [f"SLOT_{index:06d}" for index in range(1, len(slots_df) + 1)])
    return slots_df


SCENARIO_START = enriched_windows["start_time"].min().floor("min")
SCENARIO_END = enriched_windows["end_time"].max().ceil("min")
candidate_slots = generate_candidate_slots(enriched_windows, MIN_CONTACT_DURATION_SECONDS, STEP_SIZE_SECONDS)

parent_slot_counts = candidate_slots.groupby("parent_window_id").size()
print(f"Candidate slots generated: {len(candidate_slots)}")
print(f"Parent windows with at least one candidate slot: {parent_slot_counts.size}")
print(f"Parent windows with no candidate slots: {len(enriched_windows) - parent_slot_counts.size}")
print(f"Average candidate slots per generating parent window: {parent_slot_counts.mean():.2f}")
print("\nCandidate duration summary, seconds:")
print(candidate_slots["duration_seconds"].describe().to_string())
candidate_slots.head(10)

Candidate slots generated: 5231
Parent windows with at least one candidate slot: 4142
Parent windows with no candidate slots: 1920
Average candidate slots per generating parent window: 1.26

Candidate duration summary, seconds:
count    5231.0
mean      300.0
std         0.0
min       300.0
25%       300.0
50%       300.0
75%       300.0
max       300.0


,slot_id,parent_window_id,source_interval_id,satellite_name,satellite_id,norad_id,antenna_id,station_id,station_name,start_time,...,parent_visibility_end,parent_visibility_duration_seconds,candidate_index_in_parent,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,reconfiguration_time_hours,is_operationally_available
0,SLOT_000001,VIS_000006,VIS_000006,STARLINK-1039,44744,44744,ANT_GATEWAY_0005,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31T00:00:00+00:00,...,2026-05-31T00:07:00+00:00,420,1,53.684,2026-05-30T09:40:14Z,dish,1,180,0.05,True
1,SLOT_000002,VIS_000007,VIS_000007,STARLINK-1284,45390,45390,ANT_GATEWAY_0005,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31T00:00:00+00:00,...,2026-05-31T00:05:00+00:00,300,1,26.758,2026-05-30T03:35:30Z,dish,1,180,0.05,True
2,SLOT_000003,VIS_000010,VIS_000010,STARLINK-1350,45535,45535,ANT_GATEWAY_0007,GATEWAY_0007,"Anderson, SC",2026-05-31T00:00:00+00:00,...,2026-05-31T00:05:00+00:00,300,1,26.181,2026-05-30T21:51:18Z,dish,1,180,0.05,True
3,SLOT_000004,VIS_000011,VIS_000011,STARLINK-1357,45565,45565,ANT_GATEWAY_0008,GATEWAY_0008,"Angeles, Philippines",2026-05-31T00:00:00+00:00,...,2026-05-31T00:06:00+00:00,360,1,49.539,2026-05-30T20:53:36Z,dish,1,180,0.05,True
4,SLOT_000005,VIS_000017,VIS_000017,STARLINK-1246,45235,45235,ANT_GATEWAY_0011,GATEWAY_0011,"Arlington, OR",2026-05-31T00:00:00+00:00,...,2026-05-31T00:05:00+00:00,300,1,35.073,2026-05-31T01:07:36Z,dish,1,180,0.05,True
5,SLOT_000006,VIS_000022,VIS_000022,STARLINK-1350,45535,45535,ANT_GATEWAY_0014,GATEWAY_0014,"Atlanta, GA",2026-05-31T00:00:00+00:00,...,2026-05-31T00:05:00+00:00,300,1,36.397,2026-05-30T21:51:18Z,dish,1,180,0.05,True
6,SLOT_000007,VIS_000030,VIS_000030,STARLINK-1350,45535,45535,ANT_GATEWAY_0020,GATEWAY_0020,"Baxley, GA",2026-05-31T00:00:00+00:00,...,2026-05-31T00:06:00+00:00,360,1,34.355,2026-05-30T21:51:18Z,dish,1,180,0.05,True
7,SLOT_000008,VIS_000036,VIS_000036,STARLINK-1246,45235,45235,ANT_GATEWAY_0022,GATEWAY_0022,"Bellingham, WA",2026-05-31T00:00:00+00:00,...,2026-05-31T00:05:00+00:00,300,1,51.902,2026-05-31T01:07:36Z,dish,1,180,0.05,True
8,SLOT_000009,VIS_000037,VIS_000037,STARLINK-1230,45189,45189,ANT_GATEWAY_0023,GATEWAY_0023,"Benkelman, NE",2026-05-31T00:00:00+00:00,...,2026-05-31T00:06:00+00:00,360,1,38.895,2026-05-31T01:04:23Z,dish,1,180,0.05,True
9,SLOT_000010,VIS_000038,VIS_000038,STARLINK-1296,45412,45412,ANT_GATEWAY_0023,GATEWAY_0023,"Benkelman, NE",2026-05-31T00:00:00+00:00,...,2026-05-31T00:07:00+00:00,420,1,60.375,2026-05-31T04:29:06Z,dish,1,180,0.05,True


## E2 - Assign Parent Opportunity Metadata

Priority, urgency, normalized metrics, and weight are computed once per enriched visibility window. Every child candidate slot from the same parent inherits those values unchanged. This makes alternatives from the same visibility opportunity equally valuable; the optimizer chooses among them based on conflicts and feasibility, not arbitrary scoring differences.

In [4]:
def deterministic_integer(min_value: int, max_value: int, *parts: object) -> int:
    """Return a stable pseudo-random integer in a closed interval.

    Parameters
    ----------
    min_value:
        Smallest integer that can be returned.
    max_value:
        Largest integer that can be returned.
    *parts:
        Values included in the deterministic hash.

    Returns
    -------
    int
        Deterministic integer between `min_value` and `max_value`.
    """
    if min_value > max_value:
        raise ValueError("min_value cannot be greater than max_value.")

    text = "|".join(str(part) for part in parts)
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    span = max_value - min_value + 1
    return min_value + (int(digest[:12], 16) % span)


def assign_parent_priority(row: pd.Series) -> int:
    """Assign deterministic mission-like priority to one parent visibility window.

    Parameters
    ----------
    row:
        Enriched visibility-window row.

    Returns
    -------
    int
        Priority score from 1 to 5, where 5 is highest priority.
    """
    base_priority = deterministic_integer(1, 5, PRIORITY_RANDOM_SEED, row["norad_id"])
    if row["max_elevation_deg"] >= 60:
        base_priority += 1
    elif row["max_elevation_deg"] < 20:
        base_priority -= 1
    return min(5, max(1, base_priority))


def compute_parent_opportunity_metadata(parent_windows_df: pd.DataFrame) -> pd.DataFrame:
    """Compute priority, urgency, normalized metrics, and weight per parent window.

    Parameters
    ----------
    parent_windows_df:
        Enriched visibility windows that are eligible for candidate-slot generation.

    Returns
    -------
    pandas.DataFrame
        Parent-window metadata keyed by `parent_window_id` for child-slot inheritance.
    """
    parents = parent_windows_df.copy()
    horizon_seconds = max(1.0, (SCENARIO_END - SCENARIO_START).total_seconds())

    parents["priority"] = parents.apply(assign_parent_priority, axis=1)
    seconds_until_window_start = (parents["start_time"] - SCENARIO_START).dt.total_seconds().clip(lower=0)
    urgency_norm = (1.0 - (seconds_until_window_start / horizon_seconds)).clip(lower=0, upper=1)
    parents["urgency"] = (1 + (urgency_norm * 4)).round().astype(int).clip(lower=1, upper=5)

    max_visibility_duration = max(float(parents["duration_seconds"].max()), 1.0)
    parents["priority_norm"] = (parents["priority"] - 1) / 4
    parents["duration_norm"] = (parents["duration_seconds"] / max_visibility_duration).clip(lower=0, upper=1)
    parents["urgency_norm"] = (parents["urgency"] - 1) / 4
    parents["elevation_norm"] = (parents["max_elevation_deg"] / 90).clip(lower=0, upper=1)

    parents["weight"] = (
        100
        * (
            PRIORITY_WEIGHT * parents["priority_norm"]
            + DURATION_WEIGHT * parents["duration_norm"]
            + URGENCY_WEIGHT * parents["urgency_norm"]
            + ELEVATION_WEIGHT * parents["elevation_norm"]
        )
    ).round(3)

    return parents[
        [
            "window_id",
            "priority",
            "urgency",
            "priority_norm",
            "duration_norm",
            "urgency_norm",
            "elevation_norm",
            "weight",
        ]
    ].rename(columns={"window_id": "parent_window_id"})


def add_inherited_candidate_slot_weights(slots_df: pd.DataFrame, parent_windows_df: pd.DataFrame) -> pd.DataFrame:
    """Attach parent-level optimization metadata to every generated candidate slot.

    Parameters
    ----------
    slots_df:
        Candidate slots generated from enriched visibility windows.
    parent_windows_df:
        Enriched visibility windows used as candidate-slot parents.

    Returns
    -------
    pandas.DataFrame
        Candidate slots with inherited parent opportunity metadata.
    """
    parent_metadata = compute_parent_opportunity_metadata(parent_windows_df)
    slots = slots_df.merge(parent_metadata, on="parent_window_id", how="left")

    if slots["weight"].isna().any():
        raise ValueError("Some candidate slots could not inherit parent opportunity metadata.")

    slots["scenario_tag"] = "starlink_realistic_discretized_full"
    slots["priority_mode"] = PRIORITY_MODE
    slots["urgency_mode"] = URGENCY_MODE
    slots["weight_scope"] = "parent_visibility_window"
    slots["min_contact_duration_seconds"] = MIN_CONTACT_DURATION_SECONDS
    slots["discretization_step_seconds"] = STEP_SIZE_SECONDS
    return slots


weighted_slots = add_inherited_candidate_slot_weights(candidate_slots, enriched_windows)

same_parent_score_counts = weighted_slots.groupby("parent_window_id")[["priority", "urgency", "weight"]].nunique()
parents_with_varying_scores = int((same_parent_score_counts > 1).any(axis=1).sum())

print(f"Weighted candidate slots: {len(weighted_slots)}")
print(f"Parents with varying child priority/urgency/weight: {parents_with_varying_scores}")
print("Priority counts:")
print(weighted_slots["priority"].value_counts().sort_index().to_string())
print("\nUrgency counts:")
print(weighted_slots["urgency"].value_counts().sort_index().to_string())
weighted_slots.head(10)

Weighted candidate slots: 5231
Parents with varying child priority/urgency/weight: 0
Priority counts:
priority
1     515
2     996
3    1317
4    1073
5    1330

Urgency counts:
urgency
1     811
2    1001
3    1425
4    1286
5     708


,slot_id,parent_window_id,source_interval_id,satellite_name,satellite_id,norad_id,antenna_id,station_id,station_name,start_time,...,duration_norm,urgency_norm,elevation_norm,weight,scenario_tag,priority_mode,urgency_mode,weight_scope,min_contact_duration_seconds,discretization_step_seconds
0,SLOT_000001,VIS_000006,VIS_000006,STARLINK-1039,44744,44744,ANT_GATEWAY_0005,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31T00:00:00+00:00,...,0.875,1.0,0.596489,70.340,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
1,SLOT_000002,VIS_000007,VIS_000007,STARLINK-1284,45390,45390,ANT_GATEWAY_0005,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.297311,49.848,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
2,SLOT_000003,VIS_000010,VIS_000010,STARLINK-1350,45535,45535,ANT_GATEWAY_0007,GATEWAY_0007,"Anderson, SC",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.290900,83.534,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
3,SLOT_000004,VIS_000011,VIS_000011,STARLINK-1357,45565,45565,ANT_GATEWAY_0008,GATEWAY_0008,"Angeles, Philippines",2026-05-31T00:00:00+00:00,...,0.750,1.0,0.550433,89.254,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
4,SLOT_000005,VIS_000017,VIS_000017,STARLINK-1246,45235,45235,ANT_GATEWAY_0011,GATEWAY_0011,"Arlington, OR",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.389700,73.272,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
5,SLOT_000006,VIS_000022,VIS_000022,STARLINK-1350,45535,45535,ANT_GATEWAY_0014,GATEWAY_0014,"Atlanta, GA",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.404411,84.669,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
6,SLOT_000007,VIS_000030,VIS_000030,STARLINK-1350,45535,45535,ANT_GATEWAY_0020,GATEWAY_0020,"Baxley, GA",2026-05-31T00:00:00+00:00,...,0.750,1.0,0.381722,87.567,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
7,SLOT_000008,VIS_000036,VIS_000036,STARLINK-1246,45235,45235,ANT_GATEWAY_0022,GATEWAY_0022,"Bellingham, WA",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.576689,75.142,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
8,SLOT_000009,VIS_000037,VIS_000037,STARLINK-1230,45189,45189,ANT_GATEWAY_0023,GATEWAY_0023,"Benkelman, NE",2026-05-31T00:00:00+00:00,...,0.750,1.0,0.432167,76.822,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
9,SLOT_000010,VIS_000038,VIS_000038,STARLINK-1296,45412,45412,ANT_GATEWAY_0023,GATEWAY_0023,"Benkelman, NE",2026-05-31T00:00:00+00:00,...,0.875,1.0,0.670833,71.083,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120


## Build Reviewable Scheduling Instances

The small and medium instances are chronological subsets of candidate slots. The dense instance focuses on the busiest station resources, increasing same-station overlap and reconfiguration pressure for later conflict-graph experiments.

In [5]:
def finalize_instance(slots_df: pd.DataFrame, scenario_tag: str, max_rows: int) -> pd.DataFrame:
    """Create a scheduling instance with local slot IDs and a scenario tag.

    Parameters
    ----------
    slots_df:
        Weighted candidate slots to include.
    scenario_tag:
        Scenario label saved on every row.
    max_rows:
        Maximum number of rows to retain.

    Returns
    -------
    pandas.DataFrame
        Instance table sorted by start time and weight.
    """
    instance = (
        slots_df.sort_values(["start_time", "station_id", "weight"], ascending=[True, True, False])
        .head(max_rows)
        .copy()
        .reset_index(drop=True)
    )
    instance["slot_id"] = [f"{scenario_tag.upper()}_{index:05d}" for index in range(1, len(instance) + 1)]
    instance["scenario_tag"] = scenario_tag
    return instance


def build_dense_conflict_instance(slots_df: pd.DataFrame, station_count: int, max_rows: int) -> pd.DataFrame:
    """Create a denser candidate-slot instance from the busiest station resources.

    Parameters
    ----------
    slots_df:
        Full weighted candidate-slot table.
    station_count:
        Number of high-traffic stations to include.
    max_rows:
        Maximum number of rows retained in the dense instance.

    Returns
    -------
    pandas.DataFrame
        Conflict-heavy instance biased toward shared station resources.
    """
    busiest_stations = (
        slots_df.groupby("station_id")
        .size()
        .sort_values(ascending=False)
        .head(station_count)
        .index
    )
    dense_candidates = slots_df.loc[slots_df["station_id"].isin(busiest_stations)].copy()
    dense_candidates = dense_candidates.sort_values(["station_id", "start_time", "weight"], ascending=[True, True, False])
    return finalize_instance(dense_candidates, "dense_station_conflict", max_rows)


small_instance = finalize_instance(weighted_slots, "small_discretized", SMALL_INSTANCE_SIZE)
medium_instance = finalize_instance(weighted_slots, "medium_discretized", MEDIUM_INSTANCE_SIZE)
dense_instance = build_dense_conflict_instance(weighted_slots, DENSE_STATION_COUNT, DENSE_INSTANCE_SIZE)

print(f"Small instance rows: {len(small_instance)}")
print(f"Medium instance rows: {len(medium_instance)}")
print(f"Dense instance rows: {len(dense_instance)}")
print("\nDense instance station counts:")
print(dense_instance["station_name"].value_counts().head(10).to_string())

Small instance rows: 100
Medium instance rows: 500
Dense instance rows: 561

Dense instance station counts:
station_name
Cass County, ND    99
Conrad, MT         97
Arlington, OR      94
Butte, MT          93
Brewster, WA       91
Colburn, ID        87


## Validation

This section validates the new Layer E contract: enriched visibility windows are loaded, child candidate slots are generated by discretization, each child remains inside its parent window, and the final candidate slots have optimization metadata.

In [6]:
def validate_candidate_slots(slots_df: pd.DataFrame, parent_windows_df: pd.DataFrame, table_name: str) -> dict:
    """Validate a candidate-slot table generated from enriched visibility windows.

    Parameters
    ----------
    slots_df:
        Candidate-slot table to validate.
    parent_windows_df:
        Enriched visibility-window table used as the parent source.
    table_name:
        Human-readable name included in error messages.

    Returns
    -------
    dict
        Compact report for missing values, duplicate IDs, timing, and weights.
    """
    required_columns = [
        "slot_id",
        "parent_window_id",
        "satellite_name",
        "norad_id",
        "station_id",
        "station_name",
        "start_time",
        "end_time",
        "duration_seconds",
        "priority",
        "urgency",
        "weight",
        "scenario_tag",
    ]
    missing_columns = [column for column in required_columns if column not in slots_df.columns]
    if missing_columns:
        raise ValueError(f"{table_name} is missing required columns: {missing_columns}")

    parent_lookup = parent_windows_df.set_index("window_id")[["start_time", "end_time"]]
    slot_starts = pd.to_datetime(slots_df["start_time"], utc=True)
    slot_ends = pd.to_datetime(slots_df["end_time"], utc=True)
    parent_starts = slots_df["parent_window_id"].map(parent_lookup["start_time"])
    parent_ends = slots_df["parent_window_id"].map(parent_lookup["end_time"])

    same_parent_score_counts = slots_df.groupby("parent_window_id")[["priority", "urgency", "weight"]].nunique()
    parents_with_varying_child_scores = int((same_parent_score_counts > 1).any(axis=1).sum())

    report = {
        "rows": len(slots_df),
        "missing_slot_ids": int(slots_df["slot_id"].fillna("").str.strip().eq("").sum()),
        "duplicate_slot_ids": int(slots_df["slot_id"].duplicated().sum()),
        "missing_parent_window_ids": int(slots_df["parent_window_id"].fillna("").str.strip().eq("").sum()),
        "unknown_parent_window_ids": int(parent_starts.isna().sum()),
        "invalid_time_order": int((slot_ends <= slot_starts).sum()),
        "slots_outside_parent_window": int(((slot_starts < parent_starts) | (slot_ends > parent_ends)).sum()),
        "unexpected_duration": int((slots_df["duration_seconds"] != MIN_CONTACT_DURATION_SECONDS).sum()),
        "invalid_priority": int((~slots_df["priority"].between(1, 5)).sum()),
        "invalid_urgency": int((~slots_df["urgency"].between(1, 5)).sum()),
        "negative_weights": int((slots_df["weight"] < 0).sum()),
        "parents_with_varying_child_scores": parents_with_varying_child_scores,
        "missing_scenario_tag": int(slots_df["scenario_tag"].fillna("").str.strip().eq("").sum()),
    }
    failed_checks = {name: value for name, value in report.items() if name != "rows" and value != 0}
    if failed_checks:
        raise ValueError(f"{table_name} validation failed: {failed_checks}")
    return report


validation_reports = {
    "full": validate_candidate_slots(weighted_slots, enriched_windows, "full weighted candidate slots"),
    "small": validate_candidate_slots(small_instance, enriched_windows, "small instance"),
    "medium": validate_candidate_slots(medium_instance, enriched_windows, "medium instance"),
    "dense": validate_candidate_slots(dense_instance, enriched_windows, "dense instance"),
}

example_parent_id = parent_slot_counts.sort_values(ascending=False).index[0]
example_parent = enriched_windows.loc[enriched_windows["window_id"].eq(example_parent_id)]
example_children = weighted_slots.loc[weighted_slots["parent_window_id"].eq(example_parent_id)]

print("Validation reports:")
print(validation_reports)
print(f"\nEnriched visibility windows loaded: {len(enriched_windows)}")
print(f"Candidate slots generated: {len(weighted_slots)}")
print(f"Average candidate slots per loaded parent window: {len(weighted_slots) / len(enriched_windows):.2f}")
print(f"Average candidate slots per generating parent window: {parent_slot_counts.mean():.2f}")
print("\nCandidate duration summary, seconds:")
print(weighted_slots["duration_seconds"].describe().to_string())
print(f"\nExample parent window: {example_parent_id}")
print(example_parent[["window_id", "satellite_name", "station_name", "start_time", "end_time", "duration_seconds"]].to_string(index=False))
print("\nGenerated child slots for example parent:")
print(example_children[["slot_id", "parent_window_id", "start_time", "end_time", "duration_seconds", "priority", "urgency", "weight"]].to_string(index=False))
print("\nPreview of final candidate slots:")
weighted_slots.head(10)

Validation reports:
{'full': {'rows': 5231, 'missing_slot_ids': 0, 'duplicate_slot_ids': 0, 'missing_parent_window_ids': 0, 'unknown_parent_window_ids': 0, 'invalid_time_order': 0, 'slots_outside_parent_window': 0, 'unexpected_duration': 0, 'invalid_priority': 0, 'invalid_urgency': 0, 'negative_weights': 0, 'parents_with_varying_child_scores': 0, 'missing_scenario_tag': 0}, 'small': {'rows': 100, 'missing_slot_ids': 0, 'duplicate_slot_ids': 0, 'missing_parent_window_ids': 0, 'unknown_parent_window_ids': 0, 'invalid_time_order': 0, 'slots_outside_parent_window': 0, 'unexpected_duration': 0, 'invalid_priority': 0, 'invalid_urgency': 0, 'negative_weights': 0, 'parents_with_varying_child_scores': 0, 'missing_scenario_tag': 0}, 'medium': {'rows': 500, 'missing_slot_ids': 0, 'duplicate_slot_ids': 0, 'missing_parent_window_ids': 0, 'unknown_parent_window_ids': 0, 'invalid_time_order': 0, 'slots_outside_parent_window': 0, 'unexpected_duration': 0, 'invalid_priority': 0, 'invalid_urgency': 0, '

,slot_id,parent_window_id,source_interval_id,satellite_name,satellite_id,norad_id,antenna_id,station_id,station_name,start_time,...,duration_norm,urgency_norm,elevation_norm,weight,scenario_tag,priority_mode,urgency_mode,weight_scope,min_contact_duration_seconds,discretization_step_seconds
0,SLOT_000001,VIS_000006,VIS_000006,STARLINK-1039,44744,44744,ANT_GATEWAY_0005,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31T00:00:00+00:00,...,0.875,1.0,0.596489,70.340,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
1,SLOT_000002,VIS_000007,VIS_000007,STARLINK-1284,45390,45390,ANT_GATEWAY_0005,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.297311,49.848,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
2,SLOT_000003,VIS_000010,VIS_000010,STARLINK-1350,45535,45535,ANT_GATEWAY_0007,GATEWAY_0007,"Anderson, SC",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.290900,83.534,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
3,SLOT_000004,VIS_000011,VIS_000011,STARLINK-1357,45565,45565,ANT_GATEWAY_0008,GATEWAY_0008,"Angeles, Philippines",2026-05-31T00:00:00+00:00,...,0.750,1.0,0.550433,89.254,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
4,SLOT_000005,VIS_000017,VIS_000017,STARLINK-1246,45235,45235,ANT_GATEWAY_0011,GATEWAY_0011,"Arlington, OR",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.389700,73.272,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
5,SLOT_000006,VIS_000022,VIS_000022,STARLINK-1350,45535,45535,ANT_GATEWAY_0014,GATEWAY_0014,"Atlanta, GA",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.404411,84.669,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
6,SLOT_000007,VIS_000030,VIS_000030,STARLINK-1350,45535,45535,ANT_GATEWAY_0020,GATEWAY_0020,"Baxley, GA",2026-05-31T00:00:00+00:00,...,0.750,1.0,0.381722,87.567,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
7,SLOT_000008,VIS_000036,VIS_000036,STARLINK-1246,45235,45235,ANT_GATEWAY_0022,GATEWAY_0022,"Bellingham, WA",2026-05-31T00:00:00+00:00,...,0.625,1.0,0.576689,75.142,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
8,SLOT_000009,VIS_000037,VIS_000037,STARLINK-1230,45189,45189,ANT_GATEWAY_0023,GATEWAY_0023,"Benkelman, NE",2026-05-31T00:00:00+00:00,...,0.750,1.0,0.432167,76.822,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120
9,SLOT_000010,VIS_000038,VIS_000038,STARLINK-1296,45412,45412,ANT_GATEWAY_0023,GATEWAY_0023,"Benkelman, NE",2026-05-31T00:00:00+00:00,...,0.875,1.0,0.670833,71.083,starlink_realistic_discretized_full,satellite_stable_with_pass_quality,earlier_slots_more_urgent,parent_visibility_window,300,120


## Save Phase 5 Outputs

The full weighted table is the candidate-slot dataset produced by Layer E. The small, medium, and dense CSVs are reviewable optimization instances derived from those candidate slots.

In [7]:
weighted_slots.to_csv(FULL_OUTPUT_CSV, index=False)
small_instance.to_csv(SMALL_OUTPUT_CSV, index=False)
medium_instance.to_csv(MEDIUM_OUTPUT_CSV, index=False)
dense_instance.to_csv(DENSE_OUTPUT_CSV, index=False)

print(f"Saved full weighted candidate slots to: {FULL_OUTPUT_CSV.resolve()}")
print(f"Saved small instance to: {SMALL_OUTPUT_CSV.resolve()}")
print(f"Saved medium instance to: {MEDIUM_OUTPUT_CSV.resolve()}")
print(f"Saved dense instance to: {DENSE_OUTPUT_CSV.resolve()}")
print(f"Rows saved: full={len(weighted_slots)}, small={len(small_instance)}, medium={len(medium_instance)}, dense={len(dense_instance)}")

Saved full weighted candidate slots to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_slots_weighted.csv
Saved small instance to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_instance_small.csv
Saved medium instance to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_instance_medium.csv
Saved dense instance to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\starlink_scheduling_instance_dense.csv
Rows saved: full=5231, small=100, medium=500, dense=561


## Phase 5 Result

Phase 5 now outputs candidate slots, not one row per visibility window. Each candidate slot keeps `parent_window_id` and `source_interval_id` for traceability back to the enriched visibility window. The final CSVs are ready to be adapted into the existing `CandidateSlot` objects without changing the downstream optimization logic.